# Variante Kaggle — J5

Importer ce fichier dans un notebook Kaggle privé, ajouter les données officielles autorisées playground-series-s6e11, puis choisir CPU. Exécuter dans l’ordre et récupérer MyLittleMLLab/runs/J5. Les deux cellules d’amorçage précèdent les repères D1 à D12. Aucun téléchargement authentifié ni soumission automatique.

In [ ]:
# Code du laboratoire embarqué ; aucune donnée ni aucun identifiant secret.
from pathlib import Path
import os,io,zipfile,base64
OUT=Path(os.environ.get('MLLAB_KAGGLE_OUTPUT','/kaggle/working'))/'MyLittleMLLab'
OUT.mkdir(parents=True,exist_ok=True)
zipfile.ZipFile(io.BytesIO(base64.b64decode('...'))).extractall(OUT)
INPUT=Path(os.environ.get('MLLAB_KAGGLE_INPUT','/kaggle/input'))
candidates=[p.parent for p in INPUT.rglob('train.csv') if (p.parent/'test.csv').exists() and (p.parent/'sample_submission.csv').exists()]
import pandas as pd
candidates=[p for p in candidates if 'is_fraud' in pd.read_csv(p/'train.csv',nrows=1).columns]
if len(candidates)!=1:raise FileNotFoundError('Ajouter uniquement les trois CSV officiels de playground-series-s6e11 ; source absente ou ambiguë.')
folder=OUT/'data/competitions/playground-series-s6e11';folder.mkdir(parents=True,exist_ok=True)
for name in ['train.csv','test.csv','sample_submission.csv']:
 target=folder/name
 if not target.exists():target.symlink_to((candidates[0]/name).resolve())
os.chdir(OUT)
print('Laboratoire préparé ; données officielles reliées ; CPU ; aucun envoi automatique.')


# AML_05 — Choisir une expérience et son budget

Conception pédagogique : Mathieu Weill.

**QUID : Quelle amélioration mérite notre calcul ?**

Atelier guidé : lire, prévoir une sortie, exécuter au signal, expliquer la preuve. Les repères D1 à D12 sont identiques dans les diapositives et le PDF. Aucun envoi Kaggle automatique.

Télécharger le kit complet de la séance ; placer ce notebook dans `MyLittleMLLab/notebooks`, avec `mllab`, les données officielles et l’environnement. Les exemples indiqués comme fictifs sont distincts de la compétition.

[Guide étudiant](../DEMARRER_J5.md) · [PDF étudiant](../lectures/AML_05_cours.pdf)

## D1 — Retrouver le contrat

Quelle partie sert à apprendre ? Quelle partie a déjà servi à choisir ? Attendu : données officielles présentes, 3 folds fixes, CPU. En cas de fichiers absents, suivre le guide ; aucun remplacement silencieux par des données fictives.

**Au signal :** exécuter une cellule, puis nommer une valeur et sa signification. Ne pas enchaîner si l’étape précédente est en erreur.

In [ ]:
from pathlib import Path
import sys, json, os, time, hashlib, platform, subprocess
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
HERE=Path.cwd()
ROOT=next((p if (p/'mllab/competition.py').exists() else p/'student') for p in [HERE,*HERE.parents] if (p/'mllab/competition.py').exists() or (p/'student/mllab/competition.py').exists())
sys.path.insert(0,str(ROOT))
from mllab.experiments_j56 import context,make,fit_budget,tournament,add_blend,summary,freeze,versions
from sklearn.metrics import roc_auc_score
LIMIT=int(os.environ.get('MLLAB_LIMIT','15000'))
ctx=context(ROOT,LIMIT)
print('Train :',len(ctx['X']),'fraudes :',int(ctx['y'].sum()),'| folds : 3 | CPU')
print('Même source et séparation que J3. Contrôle local déjà consulté : pas un test vierge.')
plt.rcParams.update({'figure.figsize':(10,4),'font.size':13})

RUN=ROOT/'runs/J5';RUN.mkdir(parents=True,exist_ok=True)
print('Empreinte des IDs :',ctx['train_ids_sha256'][:16])

## D2 — Compter avant de lancer

Exemple fictif : 12 configurations, 3 folds et 2 seeds. Puis annoncer le budget réel de la classe : 4 candidats × 3 folds = 12 fits CV ; 2 petits fits techniques en plus. Un fit est un entraînement, pas une époque ni une prédiction.

**Au signal :** exécuter une cellule, puis nommer une valeur et sa signification. Ne pas enchaîner si l’étape précédente est en erreur.

In [ ]:
fits=fit_budget(12,3,2);minutes=fits*4
print('Exemple fictif :',fits,'fits ;',minutes,'minutes ;',divmod(minutes,60),'(h,min)')
BUDGET={'smoke_fits':2,'cv_fits':fit_budget(4,3),'total_j5':14}
assert fits==72 and minutes==288
print('Budget classe :',BUDGET)
plan={'observation':'La référence J3/J4 reste compétitive.','hypothese':'Le niveau de régularisation ou une combinaison à poids fixes peut modifier les erreurs.','fixe':['mêmes IDs','mêmes folds','ROC-AUC','CPU'],'candidats':['C=0.01','C=0.1 (référence)','C=1','boosting 80 étapes'],'arret':'2 smoke fits + 12 fits CV ; aucune nouvelle recherche de poids','limite':'pas de promesse de gain'}
(RUN/'plan_avant_calcul.json').write_text(json.dumps(plan,ensure_ascii=False,indent=2))
display(pd.Series(plan))

## D3 — Vérifier le chemin technique sur peu de lignes

Prévoir la forme : 100 probabilités, une par ligne. Ces deux fits ne servent pas à classer les modèles ; ils vérifient que le code fonctionne avant le tournoi. Le petit échantillon ne remplace pas la validation prévue.

**Au signal :** exécuter une cellule, puis nommer une valeur et sa signification. Ne pas enchaîner si l’étape précédente est en erreur.

In [ ]:
smoke_rows=ctx['folds'][0][0][:600];smoke=[]
for name in ['reference_C01','boost_80']:
 start=time.perf_counter();est=make(ctx,name).fit(ctx['X'].iloc[smoke_rows],ctx['y'].iloc[smoke_rows])
 p=est.predict_proba(ctx['X'].iloc[:100])[:,1]
 assert p.shape==(100,) and np.isfinite(p).all()
 smoke.append({'candidat':name,'lignes_train':len(smoke_rows),'predictions':len(p),'secondes':time.perf_counter()-start})
display(pd.DataFrame(smoke))
print('2 fits techniques réalisés ; aucun score de sélection calculé ici.')

## D4 — Voir les trois folds avant les modèles

Pour chaque ligne, une seule prédiction doit être produite par un modèle qui ne l’a pas apprise. Le prétraitement est lui aussi appris dans chaque fold. OOF signifie hors fold ; ces prédictions restent des données de développement.

**Au signal :** exécuter une cellule, puis nommer une valeur et sa signification. Ne pas enchaîner si l’étape précédente est en erreur.

In [ ]:
coverage=np.zeros(len(ctx['X']),int);fold_rows=[]
for k,(train,valid) in enumerate(ctx['folds'],1):
 assert not set(ctx['train'].iloc[train].id)&set(ctx['train'].iloc[valid].id)
 coverage[valid]+=1
 fold_rows.append({'fold':k,'lignes_train':len(train),'lignes_validation':len(valid),'fraudes_validation':int(ctx['y'].iloc[valid].sum())})
assert (coverage==1).all()
display(pd.DataFrame(fold_rows));print('Couverture OOF : exactement 1 prédiction par ligne.')

## D5 — Lancer le tournoi borné

Au signal seulement. Les 4 candidats utilisent exactement les mêmes folds. La régularisation est un frein sur les coefficients ; dans cette régression logistique, C plus petit signifie davantage de régularisation. Le boosting reprend les paramètres prévus ; nous ne les optimisons pas simultanément.

**Au signal :** exécuter une cellule, puis nommer une valeur et sa signification. Ne pas enchaîner si l’étape précédente est en erreur.

In [ ]:
oof,scores,fold_models=tournament(ctx,RUN)
assert len(scores)==12
print('12 fits CV terminés ; budget total J5 = 2 + 12 = 14 fits.')
display(summary(scores).round(6))
summary(scores).auc_moyenne.plot.bar(color='#167D8D',ylim=(0.5,1),ylabel='AUC moyenne CV',title='Résultats locaux : aucune valeur Kaggle publique')
plt.tight_layout();plt.show()

## D6 — Comparer les écarts fold par fold

Une moyenne seule peut cacher un résultat irrégulier. Lire les trois écarts à la référence, sans inventer une preuve statistique à partir de seulement trois folds. Les variations entre folds ne sont pas des intervalles de confiance.

**Au signal :** exécuter une cellule, puis nommer une valeur et sa signification. Ne pas enchaîner si l’étape précédente est en erreur.

In [ ]:
paired=scores.pivot(index='fold',columns='candidat',values='auc')
deltas=paired.subtract(paired.reference_C01,axis=0)
display(paired.round(6));display(deltas.round(6))
deltas.drop(columns='reference_C01').plot.bar(ylabel='Écart AUC à la référence',title='Mêmes folds : quels écarts sont réguliers ?')
plt.axhline(0,color='black',linewidth=1);plt.tight_layout();plt.show()

## D7 — Mettre des nombres sur la moyenne de modèles

Exemple fictif : les deux modèles se trompent sur des lignes différentes. Calculer une moyenne de scores ne crée pas de nouveaux labels. Le choix du seuil est distinct. Une complémentarité possible ne garantit pas un gain sur les données réelles.

**Au signal :** exécuter une cellule, puis nommer une valeur et sa signification. Ne pas enchaîner si l’étape précédente est en erreur.

In [ ]:
y_toy=np.array([0,0,1,1]);a=np.array([.1,.8,.7,.9]);b=np.array([.7,.1,.9,.8]);m=(a+b)/2
exemple=pd.DataFrame({'vrai':y_toy,'score_A':a,'score_B':b,'moyenne':m,'erreur_A':(a>=.5)!=y_toy,'erreur_B':(b>=.5)!=y_toy,'erreur_moyenne':(m>=.5)!=y_toy})
display(exemple)
print('AUC fictives A, B, moyenne :',[roc_auc_score(y_toy,p) for p in [a,b,m]])
assert np.allclose(m,[.4,.45,.8,.85])

## D8 — Essayer une seule combinaison annoncée

Moyenne fixée avant lecture : 50 % référence + 50 % boosting. Réutiliser leurs OOF ne demande aucun nouveau fit. Nous comparons la moyenne des AUC par fold, puis distinguons cette valeur de l’AUC calculée sur toutes les OOF regroupées.

**Au signal :** exécuter une cellule, puis nommer une valeur et sa signification. Ne pas enchaîner si l’étape précédente est en erreur.

In [ ]:
oof,scores=add_blend(oof,scores)
comparison=summary(scores)
display(comparison.round(6))
print('Fits supplémentaires de combinaison : 0 ; le coût futur sera deux modèles en inférence.')
print('AUC OOF regroupées (descriptif distinct) :',roc_auc_score(oof.target,oof.moyenne_50_50))
errors=pd.DataFrame({'reference':(oof.reference_C01>=.5)!=oof.target,'boosting':(oof.boost_80>=.5)!=oof.target})
display(pd.crosstab(errors.reference,errors.boosting,rownames=['Erreur référence'],colnames=['Erreur boosting']))

## D9 — Mesurer un coût avec un périmètre clair

Mesurer la prédiction sur 1000 lignes, après un passage de préparation, puis cinq répétitions. Ces durées dépendent de cette machine et ne décrivent pas le temps d’entraînement. Les modèles utilisés ici proviennent du dernier fold ; la mesure sert à comparer un périmètre déclaré.

**Au signal :** exécuter une cellule, puis nommer une valeur et sa signification. Ne pas enchaîner si l’étape précédente est en erreur.

In [ ]:
batch=ctx['X'].iloc[:1000];latency=[]
for name in ['reference_C01','boost_80','moyenne_50_50']:
 members=[fold_models['reference_C01'][-1],fold_models['boost_80'][-1]] if name=='moyenne_50_50' else [fold_models[name][-1]]
 def infer():return np.mean([model.predict_proba(batch)[:,1] for model in members],axis=0)
 infer();times=[]
 for _ in range(5):
  start=time.perf_counter();infer();times.append((time.perf_counter()-start)*1000)
 latency.append({'candidat':name,'lignes':len(batch),'repetitions':5,'mediane_ms':float(np.median(times))})
resources=pd.DataFrame(latency);resources.to_csv(RUN/'latence.csv',index=False)
display(resources.round(3))
print('Mémoire du seul tableau de probabilités :',oof.reference_C01.to_numpy().nbytes,'octets ; pas la RAM totale du processus.')

## D10 — Figer une décision reproductible

La règle de cet atelier retient la meilleure AUC moyenne CV parmi les cinq candidats, avec départage lexical en cas d’égalité exacte. Une règle métier pourrait privilégier le coût ; elle devrait être annoncée avant sélection. Ne pas changer la règle pour faire gagner le modèle préféré.

**Au signal :** exécuter une cellule, puis nommer une valeur et sa signification. Ne pas enchaîner si l’étape précédente est en erreur.

In [ ]:
decision=freeze(ctx,oof,scores,RUN)
print('Candidat retenu :',decision['selected'])
print('Fits de refit prévus en J6 :',decision['refit_fits_j6'])
display(pd.Series({k:decision[k] for k in ['metric','limit','cv_fits','smoke_fits','selection_fits_total','score_public']}))
print('Limites :',*decision['limits'],sep='\n- ')

## D11 — Transporter le même protocole vers Kaggle

Ne pas installer de GPU par réflexe. Le kit fournit KAGGLE_AML_05.ipynb : il embarque le code et attend les données officielles en entrée. Ici nous préparons le transfert sans lancer ni soumettre. Si la compétition est fermée, le calcul et le bilan peuvent rester pédagogiques.

**Au signal :** exécuter une cellule, puis nommer une valeur et sa signification. Ne pas enchaîner si l’étape précédente est en erreur.

In [ ]:
files=['plan_avant_calcul.json','protocole.json','scores_folds.csv','comparaison.csv','latence.csv','decision.json']
assert all((RUN/f).exists() for f in files)
transport={'notebook':'templates/KAGGLE_AML_05.ipynb','input':'playground-series-s6e11','accelerateur':'CPU','limit':LIMIT,'folds':3,'seed':2026,'metric':'roc_auc','train_ids_sha256':ctx['train_ids_sha256'],'data_sha256':ctx['record']['data_sha256'],'remote_execution':'non réalisée par cette cellule','score_public':None}
(RUN/'transport.json').write_text(json.dumps(transport,ensure_ascii=False,indent=2))
display(pd.Series(transport))

## D12 — Conclure sans confondre résultat et promesse

Remise : décision, comparaison, scores par fold et limite. Chaque étudiant explique une ligne du résultat. Le notebook affiche les mêmes repères que le support. Les données officielles et les sorties brutes contenant des lignes de compétition ne sont pas destinées à un dépôt public.

**Au signal :** exécuter une cellule, puis nommer une valeur et sa signification. Ne pas enchaîner si l’étape précédente est en erreur.

In [ ]:
required=['decision.json','comparaison.csv','scores_folds.csv','oof.csv','transport.json']
assert all((RUN/f).exists() for f in required)
assert oof.id.is_unique and (oof.fold.between(1,3)).all()
print('Dossier J5 complet :',RUN)
print('À expliquer : hypothèse → protocole → 14 fits → résultat → limite → décision.')
print('J6 reprendra decision.json, sans relancer la recherche de candidats.')